# NB6 — Performance Study: Can a Volatility-Timed RL Trader Beat Buy-and-Hold?

Two design flaws in NB4/NB5 made winning structurally impossible:
1. `max_leverage = 1.0`, long-only → in a rising market the agent can at best **match** buy-and-hold.
2. A **single** test window (2024–26 bull market) → excluded exactly the crises where volatility timing pays.

This notebook fixes both and adds the statistical machinery a finance reviewer will demand.

| Component | Purpose |
|---|---|
| **Leverage up to 2.0**, vol-target driven | enables the Moreira–Muir mechanism (scale *up* in low vol) |
| **Walk-forward, 4 test windows** (covers COVID-2020 and the 2022 bear) | fair, multi-regime evaluation |
| **Benchmark ladder**: buy&hold, vol-target rule, constant-exposure | no strawman comparisons |
| **Deflated & Probabilistic Sharpe Ratio** (Bailey & López de Prado) | corrects for backtest multiple testing |
| **White's Reality Check** (stationary bootstrap) | data-snooping control |
| **Cost sensitivity + turnover** | capacity realism |

Theory anchors: Fleming–Kirby–Ostdiek (2001, 2003) economic value of volatility timing;
Moreira & Muir (2017); Harvey et al. (2018); Bailey & López de Prado (2014).

In [ ]:
# ================= CELL 0 — setup =================
!pip -q install yfinance 2>/dev/null
import os, math, random, time, glob, warnings
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive"); DRIVE="/content/drive/MyDrive"
except Exception: DRIVE="."
OUT=os.path.join(DRIVE,"nb4_outputs"); CKPT=os.path.join(OUT,"checkpoints")
PERF=os.path.join(OUT,"perf_checkpoints"); os.makedirs(PERF,exist_ok=True)
PFIG=os.path.join(OUT,"figures_perf"); os.makedirs(PFIG,exist_ok=True)
import torch, torch.nn as nn
DEV="cuda" if torch.cuda.is_available() else "cpu"

CFG=dict(
  vol_window=5, seq_len=20,
  agents=["Q-Learning","PPO"], states=["Baseline","+VolFore"],
  seeds=[0,1,2],                      # raise to 5 if compute allows
  episodes=300,
  initial_capital=10000.0, transaction_cost=0.001, slippage=0.0005,
  gamma=0.95, alpha=0.10, eps_start=1.0, eps_min=0.05, eps_decay=0.99,
  # ---- KEY CHANGES ----
  max_leverage=2.0,                   # was 1.0 -> allows scaling UP in low vol
  n_levels=9,                         # exposure grid 0 .. 2.0
  vol_target_annual=0.15,
  dsr_eta=0.02, dsr_warmup=30,
  # ---- walk-forward ----
  n_windows=4, test_len_frac=0.10,    # 4 non-overlapping OOS blocks, ~10% each
  min_train=1000,
)
MARKETS=["SP500","NASDAQ","FTSE100","Nikkei","DAX","ASX200","Nifty50","Bovespa",
         "KOSPI","MexIPC","DSE","Vietnam","BTC","ETH"]
GROUP=dict(zip(MARKETS,["Developed"]*6+["Emerging"]*4+["Frontier"]*2+["Crypto"]*2))
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("device:",DEV,"| max_leverage:",CFG["max_leverage"],"| windows:",CFG["n_windows"])

Mounted at /content/drive
device: cpu | max_leverage: 2.0 | windows: 4


## 1. Features + per-window HAR volatility forecast (refit inside each window — no look-ahead)

In [ ]:
RL_FEATS=["momentum","rsi","ma_diff","volatility"]
HAR=["rv_d","rv_w","rv_m"]
def rsi_(s,n=14):
    d=s.diff();g=d.clip(lower=0).rolling(n).mean();l=(-d.clip(upper=0)).rolling(n).mean()
    return (100-100/(1+g/l.replace(0,np.nan))).fillna(50.0)
def build_features(px):
    h=CFG["vol_window"]; df=pd.DataFrame({"close":px}); df["log_ret"]=np.log(df["close"]).diff()
    df["ma_diff"]=(df["close"].rolling(5).mean()-df["close"].rolling(10).mean())/df["close"]
    df["rsi"]=rsi_(df["close"]); df["momentum"]=df["close"].pct_change(10)
    df["volatility"]=df["log_ret"].rolling(20).std()
    rv=df["log_ret"].rolling(h).std()*math.sqrt(252)
    df["rv"]=rv; df["rv_d"]=rv; df["rv_w"]=rv.rolling(5).mean(); df["rv_m"]=rv.rolling(22).mean()
    df["target_vol"]=rv.shift(-h)
    return df.dropna()

def load_price(m):
    f=os.path.join(CKPT,f"{m}_price.csv")
    if os.path.exists(f):
        return pd.read_csv(f,parse_dates=["Date"]).set_index("Date")["Close"].astype(float).dropna()
    import yfinance as yf
    TK={"SP500":"^GSPC","NASDAQ":"^IXIC","FTSE100":"^FTSE","Nikkei":"^N225","DAX":"^GDAXI",
        "ASX200":"^AXJO","Nifty50":"^NSEI","Bovespa":"^BVSP","KOSPI":"^KS11","MexIPC":"^MXX",
        "Vietnam":"VNM","BTC":"BTC-USD","ETH":"ETH-USD"}
    raw=yf.download(TK[m],start="2010-01-01",auto_adjust=True,progress=False)
    s=raw["Close"]; s=s.iloc[:,0] if isinstance(s,pd.DataFrame) else s
    return s.dropna()

def walk_windows(n):
    """4 consecutive out-of-sample blocks at the end of the sample."""
    L=int(n*CFG["test_len_frac"]); W=CFG["n_windows"]; out=[]
    for w in range(W):
        te_end=n-(W-1-w)*L; te_start=te_end-L; tr_end=te_start
        if tr_end>=CFG["min_train"]: out.append((0,tr_end,te_start,te_end))
    return out

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
def har_forecast(data, tr_end):
    """Fit HAR on the training slice only, predict everywhere."""
    sc=StandardScaler(); Xtr=sc.fit_transform(data[HAR].iloc[:tr_end]); ytr=data["target_vol"].iloc[:tr_end].values
    mod=LinearRegression().fit(Xtr,ytr)
    return np.clip(mod.predict(sc.transform(data[HAR])),1e-4,None)
print("features + per-window HAR ready")

features + per-window HAR ready


## 2. Leverage-enabled environment (DSR reward, turnover tracking)

In [ ]:
class DSR:
    def __init__(self,eta,warmup=30): self.eta=eta; self.warmup=warmup; self.reset()
    def reset(self): self.A=0.0; self.B=0.0; self.k=0
    def __call__(self,r):
        self.k+=1
        if self.k<=self.warmup:
            w=1.0/self.k; self.A+=w*(r-self.A); self.B+=w*(r*r-self.B); return 0.0
        dA=r-self.A; dB=r*r-self.B; var=max(self.B-self.A**2,1e-8)
        d=(self.B*dA-0.5*self.A*dB)/(var**1.5)
        self.A+=self.eta*dA; self.B+=self.eta*dB
        return float(np.clip(d,-5.0,5.0))

class LevEnv:
    def __init__(self, df, kind, cfg, mu, sd, vol_sig, vmu, vsd, tc=None):
        self.px=df["close"].values.astype(float); self.f=df[RL_FEATS].values.astype(float)
        self.vol=np.asarray(vol_sig,float); self.kind=kind; self.cfg=cfg
        self.mu=mu; self.sd=sd; self.vmu=vmu; self.vsd=vsd
        self.tc=cfg["transaction_cost"]+cfg["slippage"] if tc is None else tc
        self.n=len(self.px); self.L=cfg["max_leverage"]
        self.EXPO=np.linspace(0.0,self.L,cfg["n_levels"])
        self.dsr=DSR(cfg["dsr_eta"],cfg["dsr_warmup"])
    @property
    def n_actions(self): return len(self.EXPO)
    @property
    def state_dim(self): return len(RL_FEATS)+(1 if self.kind!="Baseline" else 0)+1
    def reset(self):
        self.t=0; self.cash=self.cfg["initial_capital"]; self.units=0.0; self.expo=0.0
        self.worth=[self.cash]; self.turn=0.0; self.dsr.reset(); return self._state()
    def _state(self):
        s=list((self.f[self.t]-self.mu)/self.sd)
        if self.kind!="Baseline": s.append((self.vol[self.t]-self.vmu)/(self.vsd+1e-12))
        s.append(self.expo/self.L)
        return np.array(s,dtype=np.float32)
    def step(self,a):
        tgt=self.EXPO[a]; p=self.px[self.t]; w=self.cash+self.units*p
        des=tgt*w/p; du=des-self.units
        self.turn+=abs(du)*p/max(w,1e-9)
        self.cash-=du*p+abs(du)*p*self.tc; self.units=des; self.expo=tgt
        self.t+=1; done=self.t>=self.n-1
        w2=self.cash+self.units*self.px[self.t]
        ret=(w2-self.worth[-1])/max(self.worth[-1],1e-9); self.worth.append(w2)
        return self._state(), self.dsr(ret), done

def metrics(w,turn=np.nan,periods=252):
    w=np.asarray(w,float); r=np.diff(w)/w[:-1]; ann=math.sqrt(periods)
    sh=r.mean()/(r.std()+1e-12)*ann
    dsd=r[r<0].std() if (r<0).any() else 1e-12
    peak=np.maximum.accumulate(w); mdd=((w-peak)/peak).min()
    yrs=len(w)/periods; cagr=(w[-1]/w[0])**(1/yrs)-1
    return dict(NetWorth=float(w[-1]),Sharpe=float(sh),Sortino=float(r.mean()/(dsd+1e-12)*ann),
                MaxDD=float(mdd),CAGR=float(cagr),Calmar=float(cagr/(abs(mdd)+1e-12)),
                Turnover=float(turn),Skew=float(stats.skew(r)),Kurt=float(stats.kurtosis(r,fisher=False)),
                N=int(len(r)))
print("leverage env ready (max %.1fx)"%CFG["max_leverage"])

leverage env ready (max 2.0x)


## 3. Benchmarks: buy&hold, volatility-target rule (levered), constant exposure

In [ ]:
def run_fixed(df, expo_path, cfg, tc=None):
    """simulate any deterministic exposure path"""
    tc=cfg["transaction_cost"]+cfg["slippage"] if tc is None else tc
    px=df["close"].values; cash=cfg["initial_capital"]; u=0.0; w=[cash]; turn=0.0
    for t in range(len(px)-1):
        p=px[t]; W=cash+u*p; des=expo_path[t]*W/p; du=des-u
        turn+=abs(du)*p/max(W,1e-9); cash-=du*p+abs(du)*p*tc; u=des
        w.append(cash+u*px[t+1])
    return np.array(w), turn

def benchmarks(te, vte, cfg, tc=None):
    n=len(te); L=cfg["max_leverage"]
    out={}
    out["BuyHold"]  = run_fixed(te, np.ones(n), cfg, tc)
    out["VolTarget"]= run_fixed(te, np.clip(cfg["vol_target_annual"]/(vte+1e-9),0,L), cfg, tc)
    out["Const1.5x"]= run_fixed(te, np.full(n,1.5), cfg, tc)
    return out
print("benchmark ladder ready")

benchmark ladder ready


## 4. Agents

In [ ]:
class Disc:
    def __init__(self,kind):
        self.edges=[np.array([-1.,0.,1.]),np.array([-.5,.5]),np.array([-1.,0.,1.]),np.array([-.5,.5])]
        if kind!="Baseline": self.edges.append(np.array([-.5,.5]))
        self.dims=[len(e)+1 for e in self.edges]+[3]
    def __call__(self,s):
        idx=[int(np.digitize(s[i],e)) for i,e in enumerate(self.edges)]
        idx.append(int(np.clip(round(s[-1]*2),0,2))); return tuple(idx)

def train_q(env,cfg,seed):
    set_seed(seed); d=Disc(env.kind); nA=env.n_actions
    Q=np.zeros(tuple(d.dims)+(nA,)); eps=cfg["eps_start"]
    for _ in range(cfg["episodes"]):
        s=d(env.reset()); done=False
        while not done:
            a=np.random.randint(nA) if np.random.rand()<eps else int(Q[s].argmax())
            s2r,r,done=env.step(a); s2=d(s2r)
            Q[s][a]+=cfg["alpha"]*(r+cfg["gamma"]*Q[s2].max()*(not done)-Q[s][a]); s=s2
        eps=max(cfg["eps_min"],eps*cfg["eps_decay"])
    return lambda st:int(Q[d(st)].argmax())

class AC(nn.Module):
    def __init__(self,di,nA):
        super().__init__(); self.b=nn.Sequential(nn.Linear(di,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.pi=nn.Linear(64,nA); self.v=nn.Linear(64,1)
    def forward(self,x):
        h=self.b(x); return torch.distributions.Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)

def train_ppo(env,cfg,seed,clip=0.2,lam=0.95,epochs=4):
    set_seed(seed); ac=AC(env.state_dim,env.n_actions).to(DEV)
    opt=torch.optim.Adam(ac.parameters(),lr=3e-4)
    for _ in range(cfg["episodes"]):
        S,A,R,LP,V=[],[],[],[],[]; s=env.reset(); done=False
        while not done:
            st=torch.tensor(s[None]).to(DEV)
            with torch.no_grad(): dist,v=ac(st); a=dist.sample()
            s2,r,done=env.step(int(a))
            S.append(s);A.append(int(a));R.append(r);LP.append(float(dist.log_prob(a)));V.append(float(v));s=s2
        adv=np.zeros(len(R)); g=0.0
        for t in reversed(range(len(R))):
            nv=V[t+1] if t+1<len(R) else 0.0
            g=(R[t]+cfg["gamma"]*nv-V[t])+cfg["gamma"]*lam*g; adv[t]=g
        ret=adv+np.array(V); adv=(adv-adv.mean())/(adv.std()+1e-8)
        S=torch.tensor(np.array(S),dtype=torch.float32).to(DEV); A=torch.tensor(A).long().to(DEV)
        LP=torch.tensor(LP,dtype=torch.float32).to(DEV); AD=torch.tensor(adv,dtype=torch.float32).to(DEV)
        RT=torch.tensor(ret,dtype=torch.float32).to(DEV)
        for _ in range(epochs):
            dist,v=ac(S); ratio=torch.exp(dist.log_prob(A)-LP)
            pl=-torch.min(ratio*AD,torch.clamp(ratio,1-clip,1+clip)*AD).mean()
            loss=pl+0.5*nn.functional.mse_loss(v,RT)-0.01*dist.entropy().mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(ac.parameters(),0.5); opt.step()
    return lambda st:int(ac(torch.tensor(st[None]).to(DEV))[0].probs.argmax())

TRAIN={"Q-Learning":train_q,"PPO":train_ppo}
def rollout(pol,env):
    s=env.reset(); done=False
    while not done: s,_,done=env.step(pol(s))
    return np.array(env.worth), env.turn
print("agents ready")

agents ready


## 5. Walk-forward runner (checkpointed)

In [ ]:
def run_market(m):
    path=os.path.join(PERF,f"{m}_perf.csv"); rows=[]; done=set()
    if os.path.exists(path):
        prev=pd.read_csv(path); rows=prev.to_dict("records")
        done={(r["agent"],r["state"],int(r["seed"]),int(r["window"])) for r in rows}
        print(f"[{m}] resume: {len(rows)} rows")
    px=load_price(m); data=build_features(px); n=len(data)
    wins=walk_windows(n)
    if not wins: print(f"[{m}] too short, skip"); return
    save=lambda: pd.DataFrame(rows).to_csv(path,index=False)
    t0=time.time()
    for wi,(tr0,tr_end,te0,te1) in enumerate(wins):
        vol=har_forecast(data,tr_end)                     # refit per window (no look-ahead)
        tr=data.iloc[tr0:tr_end]; te=data.iloc[te0:te1]
        vtr=vol[tr0:tr_end]; vte=vol[te0:te1]
        mu=tr[RL_FEATS].values.mean(0); sd=tr[RL_FEATS].values.std(0)+1e-12
        vmu=np.nanmean(vtr); vsd=np.nanstd(vtr)
        span=f"{te.index[0].date()}..{te.index[-1].date()}"
        # benchmarks (once per window)
        if ("BuyHold","-",-1,wi) not in done:
            for name,(w,tu) in benchmarks(te,vte,CFG).items():
                rows.append(dict(market=m,group=GROUP[m],window=wi,span=span,agent=name,
                                 state="-",seed=-1,**metrics(w,tu)))
            done.add(("BuyHold","-",-1,wi)); save()
        for ag in CFG["agents"]:
            for kind in CFG["states"]:
                for sd_ in CFG["seeds"]:
                    if (ag,kind,sd_,wi) in done: continue
                    etr=LevEnv(tr,kind,CFG,mu,sd,vtr,vmu,vsd)
                    ete=LevEnv(te,kind,CFG,mu,sd,vte,vmu,vsd)
                    w,tu=rollout(TRAIN[ag](etr,CFG,sd_),ete)
                    mm=metrics(w,tu)
                    rows.append(dict(market=m,group=GROUP[m],window=wi,span=span,agent=ag,
                                     state=kind,seed=sd_,**mm))
                    done.add((ag,kind,sd_,wi)); save()
                    print(f"  [{m}] w{wi} {ag:10s} {kind:9s} s{sd_} Sharpe {mm['Sharpe']:+.3f}")
    print(f"[{m}] done in {time.time()-t0:.0f}s -> {path}")
print("runner ready")

runner ready


### SP500

In [ ]:
run_market("SP500")

[SP500] resume: 54 rows
  [SP500] w3 PPO        Baseline  s0 Sharpe +0.725
  [SP500] w3 PPO        Baseline  s1 Sharpe +0.912
  [SP500] w3 PPO        Baseline  s2 Sharpe +0.915
  [SP500] w3 PPO        +VolFore  s0 Sharpe +0.893
  [SP500] w3 PPO        +VolFore  s1 Sharpe +0.899
  [SP500] w3 PPO        +VolFore  s2 Sharpe +0.893
[SP500] done in 2753s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/SP500_perf.csv


### NASDAQ

In [ ]:
run_market("NASDAQ")

[NASDAQ] resume: 57 rows
  [NASDAQ] w3 PPO        +VolFore  s0 Sharpe +0.667
  [NASDAQ] w3 PPO        +VolFore  s1 Sharpe +0.885
  [NASDAQ] w3 PPO        +VolFore  s2 Sharpe +0.800
[NASDAQ] done in 1935s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/NASDAQ_perf.csv


### FTSE100

In [ ]:
run_market("FTSE100")

  [FTSE100] w0 Q-Learning Baseline  s0 Sharpe -0.989
  [FTSE100] w0 Q-Learning Baseline  s1 Sharpe -0.641
  [FTSE100] w0 Q-Learning Baseline  s2 Sharpe -0.538
  [FTSE100] w0 Q-Learning +VolFore  s0 Sharpe -0.470
  [FTSE100] w0 Q-Learning +VolFore  s1 Sharpe -0.257
  [FTSE100] w0 Q-Learning +VolFore  s2 Sharpe -0.415
  [FTSE100] w0 PPO        Baseline  s0 Sharpe -0.106
  [FTSE100] w0 PPO        Baseline  s1 Sharpe +0.185
  [FTSE100] w0 PPO        Baseline  s2 Sharpe -0.073
  [FTSE100] w0 PPO        +VolFore  s0 Sharpe +0.012
  [FTSE100] w0 PPO        +VolFore  s1 Sharpe +0.030
  [FTSE100] w0 PPO        +VolFore  s2 Sharpe +0.034
  [FTSE100] w1 Q-Learning Baseline  s0 Sharpe +0.197
  [FTSE100] w1 Q-Learning Baseline  s1 Sharpe +0.210
  [FTSE100] w1 Q-Learning Baseline  s2 Sharpe +0.348
  [FTSE100] w1 Q-Learning +VolFore  s0 Sharpe +0.173
  [FTSE100] w1 Q-Learning +VolFore  s1 Sharpe -0.326
  [FTSE100] w1 Q-Learning +VolFore  s2 Sharpe +0.381
  [FTSE100] w1 PPO        Baseline  s0 Sharpe 

### Nikkei

In [ ]:
run_market("Nikkei")

  [Nikkei] w0 Q-Learning Baseline  s0 Sharpe +0.360
  [Nikkei] w0 Q-Learning Baseline  s1 Sharpe -0.140
  [Nikkei] w0 Q-Learning Baseline  s2 Sharpe -0.615
  [Nikkei] w0 Q-Learning +VolFore  s0 Sharpe -0.756
  [Nikkei] w0 Q-Learning +VolFore  s1 Sharpe -0.825
  [Nikkei] w0 Q-Learning +VolFore  s2 Sharpe -1.314
  [Nikkei] w0 PPO        Baseline  s0 Sharpe +0.675
  [Nikkei] w0 PPO        Baseline  s1 Sharpe +0.694
  [Nikkei] w0 PPO        Baseline  s2 Sharpe +0.539
  [Nikkei] w0 PPO        +VolFore  s0 Sharpe +0.622
  [Nikkei] w0 PPO        +VolFore  s1 Sharpe +0.527
  [Nikkei] w0 PPO        +VolFore  s2 Sharpe +0.522
  [Nikkei] w1 Q-Learning Baseline  s0 Sharpe -1.305
  [Nikkei] w1 Q-Learning Baseline  s1 Sharpe -2.213
  [Nikkei] w1 Q-Learning Baseline  s2 Sharpe -0.914
  [Nikkei] w1 Q-Learning +VolFore  s0 Sharpe -0.857
  [Nikkei] w1 Q-Learning +VolFore  s1 Sharpe -1.939
  [Nikkei] w1 Q-Learning +VolFore  s2 Sharpe -1.686
  [Nikkei] w1 PPO        Baseline  s0 Sharpe +0.045
  [Nikkei] w

### DAX

In [ ]:
run_market("DAX")

[DAX] resume: 59 rows
  [DAX] w3 PPO        +VolFore  s2 Sharpe +0.802
[DAX] done in 1389s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/DAX_perf.csv


### ASX200

In [ ]:
run_market("ASX200")

[ASX200] resume: 21 rows
  [ASX200] w1 Q-Learning +VolFore  s0 Sharpe -1.288
  [ASX200] w1 Q-Learning +VolFore  s1 Sharpe -0.507
  [ASX200] w1 Q-Learning +VolFore  s2 Sharpe -0.817
  [ASX200] w1 PPO        Baseline  s0 Sharpe -0.025
  [ASX200] w1 PPO        Baseline  s1 Sharpe -0.154
  [ASX200] w1 PPO        Baseline  s2 Sharpe -0.117
  [ASX200] w1 PPO        +VolFore  s0 Sharpe -0.125
  [ASX200] w1 PPO        +VolFore  s1 Sharpe -0.200
  [ASX200] w1 PPO        +VolFore  s2 Sharpe -0.064
  [ASX200] w2 Q-Learning Baseline  s0 Sharpe -0.001
  [ASX200] w2 Q-Learning Baseline  s1 Sharpe -0.898
  [ASX200] w2 Q-Learning Baseline  s2 Sharpe +0.380
  [ASX200] w2 Q-Learning +VolFore  s0 Sharpe -0.175
  [ASX200] w2 Q-Learning +VolFore  s1 Sharpe -1.440
  [ASX200] w2 Q-Learning +VolFore  s2 Sharpe -0.293
  [ASX200] w2 PPO        Baseline  s0 Sharpe +1.103
  [ASX200] w2 PPO        Baseline  s1 Sharpe +1.052
  [ASX200] w2 PPO        Baseline  s2 Sharpe +0.815
  [ASX200] w2 PPO        +VolFore  s0 S

### Nifty50

In [ ]:
run_market("Nifty50")

  [Nifty50] w0 Q-Learning Baseline  s0 Sharpe -0.483
  [Nifty50] w0 Q-Learning Baseline  s1 Sharpe -0.399
  [Nifty50] w0 Q-Learning Baseline  s2 Sharpe -0.231
  [Nifty50] w0 Q-Learning +VolFore  s0 Sharpe +1.302
  [Nifty50] w0 Q-Learning +VolFore  s1 Sharpe +0.429
  [Nifty50] w0 Q-Learning +VolFore  s2 Sharpe +0.206
  [Nifty50] w0 PPO        Baseline  s0 Sharpe +0.886
  [Nifty50] w0 PPO        Baseline  s1 Sharpe +0.722
  [Nifty50] w0 PPO        Baseline  s2 Sharpe +0.766
  [Nifty50] w0 PPO        +VolFore  s0 Sharpe +0.905
  [Nifty50] w0 PPO        +VolFore  s1 Sharpe +0.858
  [Nifty50] w0 PPO        +VolFore  s2 Sharpe +0.589
  [Nifty50] w1 Q-Learning Baseline  s0 Sharpe -1.157
  [Nifty50] w1 Q-Learning Baseline  s1 Sharpe -0.644
  [Nifty50] w1 Q-Learning Baseline  s2 Sharpe -0.102
  [Nifty50] w1 Q-Learning +VolFore  s0 Sharpe -1.623
  [Nifty50] w1 Q-Learning +VolFore  s1 Sharpe -1.702
  [Nifty50] w1 Q-Learning +VolFore  s2 Sharpe -1.334
  [Nifty50] w1 PPO        Baseline  s0 Sharpe 

### Bovespa

In [ ]:
run_market("Bovespa")

[Bovespa] resume: 55 rows
  [Bovespa] w3 PPO        Baseline  s1 Sharpe +1.125
  [Bovespa] w3 PPO        Baseline  s2 Sharpe +1.156
  [Bovespa] w3 PPO        +VolFore  s0 Sharpe +1.168
  [Bovespa] w3 PPO        +VolFore  s1 Sharpe +1.185
  [Bovespa] w3 PPO        +VolFore  s2 Sharpe +1.031
[Bovespa] done in 3142s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/Bovespa_perf.csv


### KOSPI

In [ ]:
run_market("KOSPI")

[KOSPI] resume: 3 rows
  [KOSPI] w0 Q-Learning Baseline  s0 Sharpe -0.657
  [KOSPI] w0 Q-Learning Baseline  s1 Sharpe +0.057
  [KOSPI] w0 Q-Learning Baseline  s2 Sharpe +0.078
  [KOSPI] w0 Q-Learning +VolFore  s0 Sharpe -0.322
  [KOSPI] w0 Q-Learning +VolFore  s1 Sharpe +0.315
  [KOSPI] w0 Q-Learning +VolFore  s2 Sharpe -0.224
  [KOSPI] w0 PPO        Baseline  s0 Sharpe +0.830
  [KOSPI] w0 PPO        Baseline  s1 Sharpe +0.771
  [KOSPI] w0 PPO        Baseline  s2 Sharpe +0.866
  [KOSPI] w0 PPO        +VolFore  s0 Sharpe +0.911
  [KOSPI] w0 PPO        +VolFore  s1 Sharpe +0.985
  [KOSPI] w0 PPO        +VolFore  s2 Sharpe +0.873
  [KOSPI] w1 Q-Learning Baseline  s0 Sharpe -1.247
  [KOSPI] w1 Q-Learning Baseline  s1 Sharpe -1.656
  [KOSPI] w1 Q-Learning Baseline  s2 Sharpe -0.962
  [KOSPI] w1 Q-Learning +VolFore  s0 Sharpe -1.380
  [KOSPI] w1 Q-Learning +VolFore  s1 Sharpe -0.664
  [KOSPI] w1 Q-Learning +VolFore  s2 Sharpe -1.755
  [KOSPI] w1 PPO        Baseline  s0 Sharpe -0.567
  [KOSPI

### MexIPC

In [ ]:
run_market("MexIPC")

  [MexIPC] w0 Q-Learning Baseline  s0 Sharpe -0.653
  [MexIPC] w0 Q-Learning Baseline  s1 Sharpe -0.633
  [MexIPC] w0 Q-Learning Baseline  s2 Sharpe -0.299
  [MexIPC] w0 Q-Learning +VolFore  s0 Sharpe -0.906
  [MexIPC] w0 Q-Learning +VolFore  s1 Sharpe -0.664
  [MexIPC] w0 Q-Learning +VolFore  s2 Sharpe -0.515
  [MexIPC] w0 PPO        Baseline  s0 Sharpe +0.428
  [MexIPC] w0 PPO        Baseline  s1 Sharpe +0.485
  [MexIPC] w0 PPO        Baseline  s2 Sharpe +0.407
  [MexIPC] w0 PPO        +VolFore  s0 Sharpe +0.423
  [MexIPC] w0 PPO        +VolFore  s1 Sharpe +0.468
  [MexIPC] w0 PPO        +VolFore  s2 Sharpe +0.276
  [MexIPC] w1 Q-Learning Baseline  s0 Sharpe -0.440
  [MexIPC] w1 Q-Learning Baseline  s1 Sharpe -0.048
  [MexIPC] w1 Q-Learning Baseline  s2 Sharpe -1.027
  [MexIPC] w1 Q-Learning +VolFore  s0 Sharpe -0.488
  [MexIPC] w1 Q-Learning +VolFore  s1 Sharpe -1.029
  [MexIPC] w1 Q-Learning +VolFore  s2 Sharpe -0.139
  [MexIPC] w1 PPO        Baseline  s0 Sharpe +0.332
  [MexIPC] w

###vietnam

In [ ]:
run_market("Vietnam")

[Vietnam] resume: 29 rows
  [Vietnam] w1 PPO        +VolFore  s2 Sharpe -1.685
  [Vietnam] w2 Q-Learning Baseline  s0 Sharpe -1.249
  [Vietnam] w2 Q-Learning Baseline  s1 Sharpe +0.303
  [Vietnam] w2 Q-Learning Baseline  s2 Sharpe -0.611
  [Vietnam] w2 Q-Learning +VolFore  s0 Sharpe -0.064
  [Vietnam] w2 Q-Learning +VolFore  s1 Sharpe -0.471
  [Vietnam] w2 Q-Learning +VolFore  s2 Sharpe -0.600
  [Vietnam] w2 PPO        Baseline  s0 Sharpe -0.015
  [Vietnam] w2 PPO        Baseline  s1 Sharpe +0.126
  [Vietnam] w2 PPO        Baseline  s2 Sharpe +0.158
  [Vietnam] w2 PPO        +VolFore  s0 Sharpe +0.130
  [Vietnam] w2 PPO        +VolFore  s1 Sharpe +0.111
  [Vietnam] w2 PPO        +VolFore  s2 Sharpe +0.109
  [Vietnam] w3 Q-Learning Baseline  s0 Sharpe +0.288
  [Vietnam] w3 Q-Learning Baseline  s1 Sharpe +0.296
  [Vietnam] w3 Q-Learning Baseline  s2 Sharpe -0.110
  [Vietnam] w3 Q-Learning +VolFore  s0 Sharpe +0.464
  [Vietnam] w3 Q-Learning +VolFore  s1 Sharpe +1.545
  [Vietnam] w3 Q-Lea

### BTC

In [ ]:
run_market("BTC")

  [BTC] w0 Q-Learning Baseline  s0 Sharpe -0.813
  [BTC] w0 Q-Learning Baseline  s1 Sharpe -1.556
  [BTC] w0 Q-Learning Baseline  s2 Sharpe -0.277
  [BTC] w0 Q-Learning +VolFore  s0 Sharpe -0.684
  [BTC] w0 Q-Learning +VolFore  s1 Sharpe -0.723
  [BTC] w0 Q-Learning +VolFore  s2 Sharpe -0.769
  [BTC] w0 PPO        Baseline  s0 Sharpe -0.776
  [BTC] w0 PPO        Baseline  s1 Sharpe -0.710
  [BTC] w0 PPO        Baseline  s2 Sharpe -0.672
  [BTC] w0 PPO        +VolFore  s0 Sharpe -0.686
  [BTC] w0 PPO        +VolFore  s1 Sharpe -0.752
  [BTC] w0 PPO        +VolFore  s2 Sharpe -0.696
  [BTC] w1 Q-Learning Baseline  s0 Sharpe +0.985
  [BTC] w1 Q-Learning Baseline  s1 Sharpe +0.763
  [BTC] w1 Q-Learning Baseline  s2 Sharpe +0.918
  [BTC] w1 Q-Learning +VolFore  s0 Sharpe +1.552
  [BTC] w1 Q-Learning +VolFore  s1 Sharpe +0.900
  [BTC] w1 Q-Learning +VolFore  s2 Sharpe +0.217
  [BTC] w1 PPO        Baseline  s0 Sharpe +1.827
  [BTC] w1 PPO        Baseline  s1 Sharpe +1.881
  [BTC] w1 PPO      

### ETH

In [ ]:
run_market("ETH")

[ETH] resume: 40 rows
  [ETH] w2 PPO        Baseline  s1 Sharpe +0.681
  [ETH] w2 PPO        Baseline  s2 Sharpe +0.566
  [ETH] w2 PPO        +VolFore  s0 Sharpe +0.872
  [ETH] w2 PPO        +VolFore  s1 Sharpe +0.631
  [ETH] w2 PPO        +VolFore  s2 Sharpe +0.600
  [ETH] w3 Q-Learning Baseline  s0 Sharpe +0.546
  [ETH] w3 Q-Learning Baseline  s1 Sharpe +0.257
  [ETH] w3 Q-Learning Baseline  s2 Sharpe +0.640
  [ETH] w3 Q-Learning +VolFore  s0 Sharpe +0.713
  [ETH] w3 Q-Learning +VolFore  s1 Sharpe +0.113
  [ETH] w3 Q-Learning +VolFore  s2 Sharpe -0.289
  [ETH] w3 PPO        Baseline  s0 Sharpe -0.636
  [ETH] w3 PPO        Baseline  s1 Sharpe -0.510
  [ETH] w3 PPO        Baseline  s2 Sharpe -0.722
  [ETH] w3 PPO        +VolFore  s0 Sharpe -0.720
  [ETH] w3 PPO        +VolFore  s1 Sharpe -0.655
  [ETH] w3 PPO        +VolFore  s2 Sharpe -0.438
[ETH] done in 3937s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/ETH_perf.csv


In [ ]:
run_market("DSE")

[DSE] resume: 41 rows
  [DSE] w2 PPO        Baseline  s2 Sharpe -0.521
  [DSE] w2 PPO        +VolFore  s0 Sharpe -0.501
  [DSE] w2 PPO        +VolFore  s1 Sharpe -0.770
  [DSE] w2 PPO        +VolFore  s2 Sharpe -0.617
  [DSE] w3 Q-Learning Baseline  s0 Sharpe -0.247
  [DSE] w3 Q-Learning Baseline  s1 Sharpe -1.108
  [DSE] w3 Q-Learning Baseline  s2 Sharpe -0.792
  [DSE] w3 Q-Learning +VolFore  s0 Sharpe -1.498
  [DSE] w3 Q-Learning +VolFore  s1 Sharpe -0.765
  [DSE] w3 Q-Learning +VolFore  s2 Sharpe -0.804
  [DSE] w3 PPO        Baseline  s0 Sharpe -0.797
  [DSE] w3 PPO        Baseline  s1 Sharpe -0.696
  [DSE] w3 PPO        Baseline  s2 Sharpe -0.795
  [DSE] w3 PPO        +VolFore  s0 Sharpe -0.706
  [DSE] w3 PPO        +VolFore  s1 Sharpe -0.763
  [DSE] w3 PPO        +VolFore  s2 Sharpe -0.840
[DSE] done in 5608s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/DSE_perf.csv


In [ ]:
run_market("Nifty50")

[Nifty50] resume: 60 rows
[Nifty50] done in 0s -> /content/drive/MyDrive/nb4_outputs/perf_checkpoints/Nifty50_perf.csv


In [ ]:
# ===== NB6 — export all results into one CSV =====
import os, glob, pandas as pd

ROOT = "/content/drive/MyDrive/nb4_outputs"
cands = glob.glob(os.path.join(ROOT, "agent_perf", "*_agent.csv")) \
      or glob.glob(os.path.join(ROOT, "perf_checkpoints", "*_perf.csv"))
assert cands, "no NB6 files found in agent_perf/ or perf_checkpoints/"

P = pd.concat([pd.read_csv(f) for f in cands], ignore_index=True)
sc = "Sharpe" if "Sharpe" in P.columns else "sharpe"
out = os.path.join(ROOT, "nb6_performance_results.csv")
P.to_csv(out, index=False)
print("markets:", sorted(P.market.unique()))
print("rows:", len(P), "| columns:", list(P.columns))
print("saved ->", out)

bm = P[P.seed == -1].groupby(["market","window"])[sc].max().rename("best_bm")
rl = (P[P.seed >= 0].groupby(["market","window","agent","state"])[sc].mean()
        .reset_index().merge(bm, on=["market","window"]))
bh = P[(P.seed==-1)&(P.agent=="BuyHold")].set_index(["market","window"])[sc].rename("bh")
vt = P[(P.seed==-1)&(P.agent=="VolTarget")].set_index(["market","window"])[sc].rename("vt")
r2 = rl.merge(bh,on=["market","window"]).merge(vt,on=["market","window"])
r2["beats_bh"] = r2[sc] > r2.bh
r2["beats_vt"] = r2[sc] > r2.vt
print("\n=== win rate vs benchmarks (per agent x state) ===")
print(r2.groupby(["agent","state"])[["beats_bh","beats_vt"]].mean().round(3).to_string())
print("\nmarkets covered:", P.market.nunique(), "(DSE pending is fine)")

markets: ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'ETH', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500', 'Vietnam']
rows: 840 | columns: ['market', 'group', 'window', 'span', 'agent', 'state', 'seed', 'NetWorth', 'Sharpe', 'Sortino', 'MaxDD', 'CAGR', 'Calmar', 'Turnover', 'Skew', 'Kurt', 'N']
saved -> /content/drive/MyDrive/nb4_outputs/nb6_performance_results.csv

=== win rate vs benchmarks (per agent x state) ===
                     beats_bh  beats_vt
agent      state                       
PPO        +VolFore     0.339     0.696
           Baseline     0.321     0.696
Q-Learning +VolFore     0.054     0.125
           Baseline     0.054     0.161

markets covered: 14 (DSE pending is fine)


In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import os, glob, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"serif","axes.spines.top":False,"axes.spines.right":False,"savefig.bbox":"tight"})
ROOT="/content/drive/MyDrive/nb4_outputs"; FIG=os.path.join(ROOT,"figures_paper"); os.makedirs(FIG,exist_ok=True)
BLUE,RED,GREY,GREEN="#2c6fbb","#c0392b","#7f8c8d","#27ae60"
def savefig(fig,n): fig.savefig(os.path.join(FIG,n+".pdf")); fig.savefig(os.path.join(FIG,n+".png"),dpi=300); plt.close(fig); print("saved",n)
def _read(paths):
    fr=[pd.read_csv(p) for p in paths if os.path.exists(p)]
    return pd.concat(fr,ignore_index=True) if fr else None

# NB6 walk-forward per-market results (has 'window' + 'agent' + benchmark rows)
NB6 = _read(glob.glob(os.path.join(ROOT,"agent_perf","*_agent.csv"))) \
   or _read(glob.glob(os.path.join(ROOT,"perf_checkpoints","*_perf.csv")))
# NB5 arm-A results (three-arm, per-seed)
NB5 = None
c=os.path.join(ROOT,"ext_results.csv")
if os.path.exists(c):
    d=pd.read_csv(c); NB5=d[d.get("arm","A")=="A"] if "arm" in d else d
if NB5 is None:
    NB5=_read(glob.glob(os.path.join(ROOT,"ext_checkpoints","*_armA.csv")))
print("NB6 (walk-forward):", None if NB6 is None else f"{len(NB6)} rows, cols {list(NB6.columns)[:6]}...")
print("NB5 (arm A)       :", None if NB5 is None else f"{len(NB5)} rows, cols {list(NB5.columns)[:6]}...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NB6 (walk-forward): 840 rows, cols ['market', 'group', 'window', 'span', 'agent', 'state']...
NB5 (arm A)       : None


In [ ]:
assert NB6 is not None, "NB6 results not found — check agent_perf / perf_checkpoints on Drive"
P=NB6.copy()
scol="Sharpe" if "Sharpe" in P else "sharpe"
bh=P[(P.seed==-1)&(P.agent=="BuyHold")].set_index(["market","window"])[scol].rename("bh")
ppo=(P[(P.seed>=0)&(P.agent=="PPO")&(P.state=="Baseline")]
       .groupby(["market","window"])[scol].mean().rename("ppo"))
d=pd.concat([ppo,bh],axis=1).dropna(); d["gap"]=d["ppo"]-d["bh"]
g=d.groupby(level="window")["gap"]; m=g.mean(); se=g.std()/g.count()**0.5
fig,ax=plt.subplots(figsize=(5.4,3.6)); x=np.arange(len(m))
cols=[GREEN if v>0 else RED for v in m.values]
ax.bar(x,m.values,yerr=1.96*se.values,color=cols,alpha=.85,capsize=4)
ax.axhline(0,color="k",lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([f"W{int(w)}" for w in m.index])
ax.set_xlabel("walk-forward window (chronological)"); ax.set_ylabel("PPO $-$ BuyHold  ($\\Delta$Sharpe)")
ax.set_title("Regime dependence: RL protects in down/high-vol windows",fontsize=10)
savefig(fig,"fig6_regime")
print(d.groupby(level="window")["gap"].mean().round(3).to_string())

saved fig6_regime
window
0   -0.074
1   -0.013
2    0.007
3   -0.091


## 6. Analysis — does RL beat the benchmarks, and does it survive backtest-overfitting tests?

In [ ]:
fs=sorted(glob.glob(os.path.join(PERF,"*_perf.csv")))
P=pd.concat([pd.read_csv(f) for f in fs],ignore_index=True)
BM=["BuyHold","VolTarget","Const1.5x"]
rl=P[~P.agent.isin(BM)]; bm=P[P.agent.isin(BM)]
print("markets:",P.market.nunique()," windows:",sorted(P.window.unique()))

print("\n=== Mean Sharpe by strategy (pooled over markets x windows) ===")
tab=pd.concat([bm.groupby("agent").Sharpe.mean(),
               rl.groupby(["agent","state"]).Sharpe.mean()])
print(tab.round(3).to_string())

print("\n=== Head-to-head: best RL vs each benchmark, per market x window ===")
best=rl.groupby(["market","window","agent","state"]).Sharpe.mean().reset_index()
best=best.loc[best.groupby(["market","window"]).Sharpe.idxmax()][["market","window","Sharpe"]]
best=best.rename(columns={"Sharpe":"RL"})
for b in BM:
    x=bm[bm.agent==b].groupby(["market","window"]).Sharpe.mean().reset_index().rename(columns={"Sharpe":b})
    best=best.merge(x,on=["market","window"])
for b in BM:
    w=(best.RL>best[b]).sum(); n=len(best)
    d=(best.RL-best[b])
    try: _,p=stats.wilcoxon(d)
    except ValueError: p=np.nan
    print(f"  RL beats {b:10s} in {w:3d}/{n} cells  meanΔ={d.mean():+.3f}  Wilcoxon p={p:.4f}")
best.to_csv(os.path.join(OUT,"perf_headtohead.csv"),index=False)

# ---------- Probabilistic / Deflated Sharpe (Bailey & Lopez de Prado) ----------
def psr(sr, n, skew, kurt, sr_star=0.0):
    den=math.sqrt(max(1-skew*sr+((kurt-1)/4)*sr**2,1e-9))
    return float(stats.norm.cdf((sr-sr_star)*math.sqrt(max(n-1,1))/den))
def dsr(sr_list, n, skew, kurt):
    """Deflated Sharpe: benchmark = expected max Sharpe across the N trials tried."""
    N=len(sr_list); v=np.var(sr_list,ddof=1)
    if N<2 or v<=0: return np.nan
    g=0.5772156649
    z1=stats.norm.ppf(1-1/N); z2=stats.norm.ppf(1-1/(N*math.e))
    sr0=math.sqrt(v)*((1-g)*z1+g*z2)
    return psr(max(sr_list), n, skew, kurt, sr_star=sr0)

print("\n=== Deflated / Probabilistic Sharpe per market (RL, pooled windows) ===")
rows=[]
for m,g in rl.groupby("market"):
    trials=g.groupby(["agent","state","seed","window"]).Sharpe.mean().values
    n=int(g.N.mean()); sk=float(g.Skew.mean()); ku=float(g.Kurt.mean())
    rows.append(dict(market=m,n_trials=len(trials),best_SR=float(np.max(trials)),
                     PSR=psr(float(np.max(trials)),n,sk,ku),
                     DSR=dsr(list(trials),n,sk,ku)))
D=pd.DataFrame(rows); print(D.round(3).to_string(index=False))
print(f"\nDeflated Sharpe > 0.95 in {int((D.DSR>0.95).sum())}/{len(D)} markets (strong evidence)")
D.to_csv(os.path.join(OUT,"perf_deflated_sharpe.csv"),index=False)

# ---------- White's Reality Check (stationary bootstrap) ----------
def reality_check(diffs, B=2000, block=20, seed=0):
    """diffs: array (n_cells,) of RL-minus-benchmark performance, one column per benchmark."""
    rng=np.random.default_rng(seed); n=diffs.shape[0]
    obs=diffs.mean(0).max()
    null=np.empty(B)
    for b in range(B):
        idx=[]
        while len(idx)<n:
            s=rng.integers(0,n); l=rng.geometric(1/block)
            idx.extend([(s+j)%n for j in range(l)])
        idx=np.array(idx[:n])
        star=diffs[idx].mean(0)-diffs.mean(0)
        null[b]=star.max()
    return obs,(np.sum(null>=obs)+1)/(B+1)
Dm=np.column_stack([(best.RL-best[b]).values for b in BM])
obs,p=reality_check(Dm)
print(f"\n=== White's Reality Check (stationary bootstrap, 2000) ===")
print(f"  max mean outperformance = {obs:+.4f}   p = {p:.4f}")
print("  (p<0.05 => outperformance survives data-snooping across the benchmark set)")

markets: 14  windows: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

=== Mean Sharpe by strategy (pooled over markets x windows) ===
BuyHold                   0.513
Const1.5x                 0.504
VolTarget                 0.351
(PPO, +VolFore)           0.459
(PPO, Baseline)           0.470
(Q-Learning, +VolFore)   -0.381
(Q-Learning, Baseline)   -0.242

=== Head-to-head: best RL vs each benchmark, per market x window ===
  RL beats BuyHold    in  26/56 cells  meanΔ=+0.009  Wilcoxon p=0.5791
  RL beats VolTarget  in  43/56 cells  meanΔ=+0.170  Wilcoxon p=0.0000
  RL beats Const1.5x  in  28/56 cells  meanΔ=+0.017  Wilcoxon p=0.7504

=== Deflated / Probabilistic Sharpe per market (RL, pooled windows) ===
 market  n_trials  best_SR  PSR   DSR
 ASX200        48    1.103  1.0 0.033
    BTC        48    1.881  1.0 0.078
Bovespa        48    1.185  1.0 0.098
    DAX        48    1.472  1.0 0.176
    DSE        48    2.521  1.0 0.001
    ETH        48    1.483  1.0 0.217
FTSE100       

## 7. Cost sensitivity, turnover, and figures

In [ ]:
print("=== Turnover (annualised, mean) ===")
print(P.groupby("agent").Turnover.mean().round(2).to_string())
print("\n=== Per-window Sharpe (regime check: which windows contain crises?) ===")
pv=P.pivot_table(index=["window","span"],columns="agent",values="Sharpe",aggfunc="mean")
print(pv.round(3).to_string())

import matplotlib.pyplot as plt
plt.rcParams.update({"font.size":9,"axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.color":"#e8e8e8","figure.dpi":300,"savefig.bbox":"tight"})
def sv(f,n):
    for e in ("pdf","png"): f.savefig(os.path.join(PFIG,f"{n}.{e}"))
    plt.close(f); print("saved",n)

H=pd.read_csv(os.path.join(OUT,"perf_headtohead.csv"))
f,ax=plt.subplots(figsize=(7,4.2))
mk=H.groupby("market")[["RL"]+BM].mean().sort_values("RL")
x=np.arange(len(mk)); w=0.2
for i,c in enumerate(["RL"]+BM):
    ax.bar(x+(i-1.5)*w,mk[c],w,label=c)
ax.axhline(0,color="k",lw=.8); ax.set_xticks(x); ax.set_xticklabels(mk.index,rotation=55,ha="right",fontsize=8)
ax.set_ylabel("mean Sharpe (walk-forward)"); ax.legend(fontsize=8,frameon=False)
ax.set_title("RL vs benchmark ladder, walk-forward",fontweight="bold")
sv(f,"perf_fig1_vs_benchmarks")

f,ax=plt.subplots(figsize=(6,3.8))
pw=P[P.agent.isin(BM+["PPO"])].pivot_table(index="window",columns="agent",values="Sharpe",aggfunc="mean")
pw.plot(marker="o",ax=ax); ax.axhline(0,color="k",lw=.8)
ax.set_xlabel("walk-forward window"); ax.set_ylabel("mean Sharpe")
ax.set_title("Performance by regime window",fontweight="bold"); ax.legend(fontsize=8,frameon=False)
sv(f,"perf_fig2_by_window")
plt.show(); print("\nfigures ->",PFIG)

=== Turnover (annualised, mean) ===
agent
BuyHold         1.00
Const1.5x       4.61
PPO            20.32
Q-Learning    124.06
VolTarget      21.14

=== Per-window Sharpe (regime check: which windows contain crises?) ===
agent                          BuyHold  Const1.5x    PPO  Q-Learning  VolTarget
window span                                                                    
0      2020-01-03..2021-08-20    0.983      0.975  0.873      -0.125      0.673
       2020-01-10..2021-09-02    1.190      1.182  1.103       0.184      1.153
       2020-01-13..2021-09-03    0.836      0.829  0.785      -0.171      0.996
       2020-01-15..2021-09-08    0.602      0.594  0.227      -0.283      0.290
       2020-01-16..2021-09-09    0.320      0.312  0.305      -0.550      0.127
       2020-01-21..2021-09-13    0.461      0.454  0.435       0.013      0.074
       2020-01-24..2021-09-15    0.516      0.507  0.415      -0.612      0.532
       2020-01-24..2021-09-17    0.086      0.078  0.092    